In [38]:
import json
import os
from typing import Any

import polars as pl

In [39]:
# PROCESS RIKISHI

base_dir_rikishi = "raw_data/rikishi"

all_item_keys_rikishi = set(
    [
        "id",
        "sumodbId",
        "nskId",
        "shikonaEn",
        "heya",
        "birthDate",
        "shusshin",
        "height",
        "weight",
        "debut",
        "intai",
        "shikonaJp",
        "currentRank",
        "updatedAt",
        "createdAt",
    ]
)

all_data_rikishi: list[dict[str, Any]] = []
file_names_rikishi = os.listdir(base_dir_rikishi)
file_names_rikishi.sort()
for fn in file_names_rikishi:
    with open(os.path.join(base_dir_rikishi, fn)) as f:
        data = json.load(f)
        if "records" in data:
            items = data["records"]
            for item in items:
                keys = set(item.keys())
                if keys != all_item_keys_rikishi:
                    print(f"{fn} does not have consistent keys")
                    missing_keys = all_item_keys_rikishi.difference(keys)
                    if missing_keys:
                        print(f"- missing: {missing_keys}")
                    extra_keys = keys.difference(all_item_keys_rikishi)
                    if extra_keys:
                        print(f"- extra: {extra_keys}")
            all_data_rikishi += items
        else:
            print(f"{fn} does not contain 'records' key.")

schema_rikishi: dict[str, pl.DataType] = {
    "id": pl.Int64,
    "shikonaEn": pl.String,
    "birthDate": pl.String,
    "shusshin": pl.String,
    "debut": pl.String,
    "intai": pl.String,
}  # type: ignore

df_rikishi = pl.from_dicts(all_data_rikishi, schema=schema_rikishi)

df_rikishi = df_rikishi.with_columns(
    pl.col("birthDate").replace("0001-01-01T00:00:00Z", None),
    pl.col("intai").replace("0001-01-01T00:00:00Z", None),
)
df_rikishi = df_rikishi.with_columns(
    pl.col("birthDate").str.to_date("%Y-%m-%dT%H:%M:%SZ"),
    pl.col("intai").str.to_date("%Y-%m-%dT%H:%M:%SZ"),
)

df_rikishi

0_1000.json does not have consistent keys
- missing: {'createdAt', 'updatedAt', 'shikonaJp', 'currentRank'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'updatedAt', 'shikonaJp', 'currentRank'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not have consistent keys
- missing: {'createdAt', 'intai'}
0_1000.json does not hav

id,shikonaEn,birthDate,shusshin,debut,intai
i64,str,date,str,str,date
1,"""Takakeisho Takanobu""",1996-08-05,"""Hyogo-ken, Ashiya-shi""","""201409""",2024-09-01
2,"""Asanoyama""",1994-03-01,"""Toyama-ken, Toyama-shi""","""201603""",null
3,"""Hakuoho""",2003-08-22,"""Tottori-ken, Kurayoshi-shi""","""202301""",null
4,"""Kaizen""",1993-07-29,"""Kagoshima-ken, Oshima-gun, Tok…","""201601""",null
5,"""Takerufuji""",1999-04-09,"""Aomori-ken, Goshogawara-shi""","""202209""",null
…,…,…,…,…,…
9047,"""Uchiumi""",2003-04-17,"""Aichi""","""202503""",null
9048,"""Akenoyama""",2006-06-13,"""Mie""","""202503""",null
9049,"""Asakawasumi""",2008-04-09,"""Tokyo""","""202503""",null


In [40]:
# PROCESS BASHO

base_dir_basho = "raw_data/basho"

all_item_keys_basho = set(
    [
        "id",
        "bashoId",
        "division",
        "day",
        "matchNo",
        "eastId",
        "eastShikona",
        "eastRank",
        "westId",
        "westShikona",
        "westRank",
        "kimarite",
        "winnerId",
        "winnerEn",
        "winnerJp",
    ]
)

all_data_basho: list[dict[str, Any]] = []
file_names_basho = os.listdir(base_dir_basho)
file_names_basho.sort()
for fn in file_names_basho:
    with open(os.path.join(base_dir_basho, fn)) as f:
        data = json.load(f)
        if "torikumi" in data:
            items = data["torikumi"]
            for item in items:
                keys = set(item.keys())
                if keys != all_item_keys_basho:
                    print(f"{fn} does not have consistent keys")
                    missing_keys = all_item_keys_basho.difference(keys)
                    if missing_keys:
                        print(f"missing keys: {missing_keys}")
                    extra_keys = keys.difference(all_item_keys_basho)
                    if extra_keys:
                        print(f"extra keys: {extra_keys}")
            all_data_basho += items
        else:
            print(f"{fn} does not contain 'torikumi' key.")


schema_basho: dict[str, pl.DataType] = {
    "id": pl.String,
    "bashoId": pl.String,
    "division": pl.String,
    "day": pl.UInt8,
    "matchNo": pl.UInt8,
    "eastId": pl.Int64,
    "eastRank": pl.String,
    "westId": pl.Int64,
    "westRank": pl.String,
    "kimarite": pl.String,
    "winnerId": pl.Int64,
}  # type: ignore

df_basho = pl.from_dicts(all_data_basho, schema=schema_basho)

year_month_from_id = pl.col("id").str.slice(0, 6).str.to_datetime("%Y%m")

# this is necessarily kinda approximate
# tournaments _usually_ start on the second Sunday of the month,
# but occasionally start on the first or third Sunday
df_basho = df_basho.insert_column(
    1,
    (
        year_month_from_id
        + pl.duration(days=14)
        - pl.duration(days=year_month_from_id.dt.weekday())
        + pl.duration(days=pl.col("day") - 1)
    )
    .dt.date()
    .alias("date"),
)

df_basho

195803_jonidan_01.json does not contain 'torikumi' key.
195803_jonidan_02.json does not contain 'torikumi' key.
195803_jonidan_03.json does not contain 'torikumi' key.
195803_jonidan_04.json does not contain 'torikumi' key.
195803_jonidan_05.json does not contain 'torikumi' key.
195803_jonidan_06.json does not contain 'torikumi' key.
195803_jonidan_07.json does not contain 'torikumi' key.
195803_jonidan_08.json does not contain 'torikumi' key.
195803_jonidan_09.json does not contain 'torikumi' key.
195803_jonidan_10.json does not contain 'torikumi' key.
195803_jonidan_11.json does not contain 'torikumi' key.
195803_jonidan_12.json does not contain 'torikumi' key.
195803_jonidan_13.json does not contain 'torikumi' key.
195803_jonidan_14.json does not contain 'torikumi' key.
195803_jonidan_15.json does not contain 'torikumi' key.
195803_jonokuchi_01.json does not contain 'torikumi' key.
195803_jonokuchi_02.json does not contain 'torikumi' key.
195803_jonokuchi_03.json does not contain 't

id,date,bashoId,division,day,matchNo,eastId,eastRank,westId,westRank,kimarite,winnerId
str,date,str,str,u8,u8,i64,str,i64,str,str,i64
"""195803-1-1311-1325""",1958-03-09,"""195803""","""Juryo""",1,1,1311,"""Makushita 1 East""",1325,"""Juryo 24 West""","""uchigake""",1311
"""195803-1-1320-1306""",1958-03-09,"""195803""","""Juryo""",1,2,1320,"""Juryo 23 East""",1306,"""Juryo 24 East""","""yorikiri""",1320
"""195803-1-1313-1309""",1958-03-09,"""195803""","""Juryo""",1,3,1313,"""Juryo 23 West""",1309,"""Juryo 22 West""","""kotenage""",1309
"""195803-1-1319-1326""",1958-03-09,"""195803""","""Juryo""",1,4,1319,"""Juryo 22 East""",1326,"""Juryo 21 West""","""tsuridashi""",1319
"""195803-1-1310-1315""",1958-03-09,"""195803""","""Juryo""",1,5,1310,"""Juryo 21 East""",1315,"""Juryo 20 West""","""makiotoshi""",1315
…,…,…,…,…,…,…,…,…,…,…,…
"""202503-15-19-184-215""",2025-03-23,"""202503""","""Sandanme""",15,20,184,"""Sandanme 12 East""",215,"""Sandanme 15 East""","""yoritaoshi""",184
"""202503-15-20-160-153""",2025-03-23,"""202503""","""Sandanme""",15,21,160,"""Sandanme 17 West""",153,"""Sandanme 11 West""","""yorikiri""",153
"""202503-15-21-500-119""",2025-03-23,"""202503""","""Sandanme""",15,22,500,"""Sandanme 13 West""",119,"""Sandanme 9 West""","""yoritaoshi""",119


In [41]:
df_rikishi_east = df_rikishi.clone().rename(
    {
        "shikonaEn": "shikonaEn_east",
        "birthDate": "birthDate_east",
        "shusshin": "shusshin_east",
        "debut": "debut_east",
        "intai": "intai_east",
    }
)
df_rikishi_west = df_rikishi.clone().rename(
    {
        "shikonaEn": "shikonaEn_west",
        "birthDate": "birthDate_west",
        "shusshin": "shusshin_west",
        "debut": "debut_west",
        "intai": "intai_west",
    }
)


df = df_basho
df = df.join(df_rikishi_east, left_on="eastId", right_on="id", how="left")
df = df.join(df_rikishi_west, left_on="westId", right_on="id", how="left")

df.write_parquet("processed_data/torikumi.parquet")

df

id,date,bashoId,division,day,matchNo,eastId,eastRank,westId,westRank,kimarite,winnerId,shikonaEn_east,birthDate_east,shusshin_east,debut_east,intai_east,shikonaEn_west,birthDate_west,shusshin_west,debut_west,intai_west
str,date,str,str,u8,u8,i64,str,i64,str,str,i64,str,date,str,str,date,str,date,str,str,date
"""195803-1-1311-1325""",1958-03-09,"""195803""","""Juryo""",1,1,1311,"""Makushita 1 East""",1325,"""Juryo 24 West""","""uchigake""",1311,"""Kamanishiki Tamenosuke""",1929-07-01,"""Kumamoto-ken, Ashikita-gun""","""194810""",1958-05-01,"""Fujinishiki Akira""",1937-03-18,"""Yamanashi-ken, Kofu-shi""","""195303""",1968-11-01
"""195803-1-1320-1306""",1958-03-09,"""195803""","""Juryo""",1,2,1320,"""Juryo 23 East""",1306,"""Juryo 24 East""","""yorikiri""",1320,"""Wakasugiyama Toyoichi""",1933-01-24,"""Fukuoka-ken, Kasuya-gun, Shime…","""195303""",1967-05-01,"""Mikasayama Mamoru""",1935-01-02,"""Hokkaido, Mikasa-shi""","""195205""",1958-09-01
"""195803-1-1313-1309""",1958-03-09,"""195803""","""Juryo""",1,3,1313,"""Juryo 23 West""",1309,"""Juryo 22 West""","""kotenage""",1309,"""Tatekabuto Sachio""",1924-10-27,"""Ehime-ken, Ochi-gun""","""194201""",1958-07-01,"""Tokitsuumi Masao""",1935-05-21,"""Toyama-ken, Himi-shi""","""195105""",1960-03-01
"""195803-1-1319-1326""",1958-03-09,"""195803""","""Juryo""",1,4,1319,"""Juryo 22 East""",1326,"""Juryo 21 West""","""tsuridashi""",1319,"""Kashiwado Tsuyoshi""",1938-11-29,"""Yamagata-ken, Higashitagawa-gu…","""195409""",1969-07-01,"""Asanishiki Toshinori""",1934-07-27,"""Aomori-ken, Minamitsugaru-gun""","""195109""",1959-05-01
"""195803-1-1310-1315""",1958-03-09,"""195803""","""Juryo""",1,5,1310,"""Juryo 21 East""",1315,"""Juryo 20 West""","""makiotoshi""",1315,"""Isenishiki Kanjiro""",1929-10-18,"""Mie-ken, Yokkaichi-shi""","""194905""",1958-09-01,"""Fukudayama Takehiro""",1931-07-18,"""Nagasaki-ken, Isahaya-shi""","""194905""",1965-01-01
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""202503-15-19-184-215""",2025-03-23,"""202503""","""Sandanme""",15,20,184,"""Sandanme 12 East""",215,"""Sandanme 15 East""","""yoritaoshi""",184,"""Chiyonoo""",1991-05-29,"""Okinawa-ken, Okinawa-shi - Kag…","""201003""",null,"""Kiyota""",2003-08-04,"""Fukushima-ken, Koriyama-shi""","""201903""",null
"""202503-15-20-160-153""",2025-03-23,"""202503""","""Sandanme""",15,21,160,"""Sandanme 17 West""",153,"""Sandanme 11 West""","""yorikiri""",153,"""Oginohama""",1994-10-27,"""Kanagawa-ken, Yokohama-shi, To…","""201003""",null,"""Ryusei""",1986-07-17,"""Tokyo-to, Katsushika-ku""","""200303""",null
"""202503-15-21-500-119""",2025-03-23,"""202503""","""Sandanme""",15,22,500,"""Sandanme 13 West""",119,"""Sandanme 9 West""","""yoritaoshi""",119,"""Takaarashi""",2006-12-28,"""Saitama-ken, Iruma-shi""","""202203""",null,"""Akitoba""",1999-08-02,"""Aichi-ken, Konan-shi""","""201503""",null
